In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm


In [23]:
from PIL import Image, UnidentifiedImageError, ImageFile
import os
import numpy as np
from tqdm import tqdm

# Allow PIL to load partially corrupted images (optional)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Path to the folder
img_folder = "C:/Users/kanch/OneDrive/Documents/MATH331/CNN/cnn_stars"

# Image settings
img_size = (128, 128)

images = []
labels = []

# Loop through files
for filename in tqdm(os.listdir(img_folder)):
    if filename.endswith(".png"):
        label = int(filename.split("_")[-1].split(".")[0])  # Extract label from filename
        img_path = os.path.join(img_folder, filename)

        try:
            # Load image, convert to grayscale, resize
            img = Image.open(img_path).convert('L')  # 'L' = grayscale
            img = img.resize(img_size)
            img_array = np.array(img) / 255.0  # Normalize to [0,1]

            images.append(img_array)
            labels.append(label)

        except (OSError, UnidentifiedImageError) as e:
            print(f"Skipping file {filename}: {e}")

# Convert to numpy arrays
X = np.array(images).reshape(-1, img_size[0], img_size[1], 1)
y = np.array(labels)


 24%|██▎       | 1342/5660 [01:39<04:34, 15.72it/s]

Skipping file star_2205_label_0.png: cannot identify image file 'C:\\Users\\kanch\\OneDrive\\Documents\\MATH331\\CNN\\cnn_stars\\star_2205_label_0.png'


 80%|████████  | 4551/5660 [05:49<01:41, 10.88it/s]

Skipping file star_5095_label_0.png: cannot identify image file 'C:\\Users\\kanch\\OneDrive\\Documents\\MATH331\\CNN\\cnn_stars\\star_5095_label_0.png'


 82%|████████▏ | 4613/5660 [05:55<01:14, 14.14it/s]

Skipping file star_5150_label_0.png: cannot identify image file 'C:\\Users\\kanch\\OneDrive\\Documents\\MATH331\\CNN\\cnn_stars\\star_5150_label_0.png'


100%|██████████| 5660/5660 [07:21<00:00, 12.83it/s]


In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [25]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_size[0], img_size[1], 1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')  
])


C:\Users\kanch\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(img_size[0], img_size[1], 1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),


    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')  
])


In [26]:
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with early stopping
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop])


Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 65s 507ms/step - accuracy: 0.9907 - loss: 0.1202 - val_accuracy: 0.9912 - val_loss: 0.0517
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 55s 483ms/step - accuracy: 0.9925 - loss: 0.0571 - val_accuracy: 0.9912 - val_loss: 0.0657
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 55s 483ms/step - accuracy: 0.9901 - loss: 0.0807 - val_accuracy: 0.9912 - val_loss: 0.0757
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 55s 478ms/step - accuracy: 0.9934 - loss: 0.0562 - val_accuracy: 0.9912 - val_loss: 0.0506
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 45s 398ms/step - accuracy: 0.9905 - loss: 0.0622 - val_accuracy: 0.9912 - val_loss: 0.0543
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 44s 383ms/step - accuracy: 0.9909 - loss: 0.0802 - val_accuracy: 0.9912 - val_loss: 0.0611
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 52s 457ms/step - accuracy: 0.9945 - loss: 0.0434 - val_accuracy: 0.9912 - val_loss: 0.0512


In [27]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")


36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - accuracy: 0.9913 - loss: 0.0499
Test Accuracy: 0.9920
